In [1]:
import sys
print(sys.executable)

/home/piets/sirna_shrna_offtargets/.venv/bin/python


In [2]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.append("../../src")
from offtarget.sequences import build_sirna_table
from offtarget.sequences import build_sirna_table, count_sites, sites_to_gene

print("All imports successful")

All imports successful


#### Assess siRNA sequences, seeds, and 3'UTR matching (from TargetScan)

In [3]:
## siRNA sense strands (RNA 5'->3') and derived guide, seeds and target sites
# siPARP1-B corrected: deposited GAUUUCAUCUGGUGUGAUA has its last 2 nt swapped
# (see CLAUDE.md and config/GSE114273.yaml)

sirna_sense = {
    "siPARP1-A": "CCAUUGAGCACUUCAUGAA",
    "siPARP1-B": "GAUUUCAUCUGGUGUGAAU",
    "siPARP2-A": "GAAGAAAUUCCUUGACAAA",
    "siPARP2-B": "AAGUACUAUCUGAUUCAGCUA",
}

sirna_table = build_sirna_table(sirna_sense)
sirna_table

,sirna_sense,sirna_guide,sirna_seed,sirna_seed_m8,site_8mer,site_7mer_m8,site_7mer_A1,site_6mer
sirna,,,,,,,,
siPARP1-A,CCAUUGAGCACUUCAUGAA,UUCAUGAAGUGCUCAAUGG,UCAUGA,UCAUGAA,TTCATGAA,TTCATGA,TCATGAA,TCATGA
siPARP1-B,GAUUUCAUCUGGUGUGAAU,AUUCACACCAGAUGAAAUC,UUCACA,UUCACAC,GTGTGAAA,GTGTGAA,TGTGAAA,TGTGAA
siPARP2-A,GAAGAAAUUCCUUGACAAA,UUUGUCAAGGAAUUUCUUC,UUGUCA,UUGUCAA,TTGACAAA,TTGACAA,TGACAAA,TGACAA
siPARP2-B,AAGUACUAUCUGAUUCAGCUA,UAGCUGAAUCAGAUAGUACUU,AGCUGA,AGCUGAA,TTCAGCTA,TTCAGCT,TCAGCTA,TCAGCT


In [4]:
## Load human ORF sequences (TargetScan, taxon 9606, gaps stripped, uppercase)
orf_path = Path("../../data/reference/processed/human_ORFs_clean.tsv.gz")
orf_tx = pd.read_csv(orf_path, sep="\t")

print("ORF transcripts:", orf_tx.shape[0])

ORF transcripts: 28351


In [5]:
## Strand check: sense should occur in its intended target ORF, guide should occur in no ORF
intended_target = {"siPARP1-A": "PARP1", "siPARP1-B": "PARP1",
                   "siPARP2-A": "PARP2", "siPARP2-B": "PARP2"}

rows = []
for name in sirna_table.index:
    for strand in ["sirna_sense", "sirna_guide"]:
        query = sirna_table.loc[name, strand].replace("U", "T")   # RNA -> DNA, to match the ORFs
        hits = orf_tx[orf_tx["orf_sequence"].str.contains(query, regex=False)]
        rows.append({
            "sirna": name,
            "strand": strand,
            "n_transcripts": len(hits),
            "transcripts": ", ".join(hits["transcript_id"]),
            "intended_target": intended_target[name],
        })

orf_check = pd.DataFrame(rows)
orf_check

,sirna,strand,n_transcripts,transcripts,intended_target
0,siPARP1-A,sirna_sense,1,ENST00000366794.5,PARP1
1,siPARP1-A,sirna_guide,0,,PARP1
2,siPARP1-B,sirna_sense,1,ENST00000366794.5,PARP1
3,siPARP1-B,sirna_guide,0,,PARP1
4,siPARP2-A,sirna_sense,2,"ENST00000429687.3, ENST00000527915.1",PARP2
5,siPARP2-A,sirna_guide,0,,PARP2
6,siPARP2-B,sirna_sense,2,"ENST00000429687.3, ENST00000527915.1",PARP2
7,siPARP2-B,sirna_guide,0,,PARP2


In [6]:
## Load human 3'UTRs (TargetScan, taxon 9606, gaps stripped, uppercase)
# One row per transcript; a gene can have several UTR isoforms (rule applied at gene aggregation)
utr_path = Path("../../data/reference/processed/human_3UTRs_clean.tsv.gz")
utr_tx = pd.read_csv(utr_path, sep="\t")

In [7]:
# Check

print("Shape:", utr_tx.shape)
print("Duplicate transcript IDs:", utr_tx["transcript_id"].duplicated().sum())
print("Missing values:", utr_tx.isna().sum().sum())
print("Unique genes:", utr_tx["gene_id_clean"].nunique())
print("Non-ACGT sequences:", (~utr_tx["utr_sequence"].str.fullmatch("[ACGT]+")).sum())
print("UTR transcripts per gene:", utr_tx.groupby("gene_id_clean").size().value_counts().sort_index().to_dict())
utr_tx.head()

Shape: (28351, 6)
Duplicate transcript IDs: 0
Missing values: 0
Unique genes: 19473
Non-ACGT sequences: 0
UTR transcripts per gene: {1: 13266, 2: 4378, 3: 1268, 4: 381, 5: 117, 6: 39, 7: 16, 8: 2, 9: 6}


,transcript_id,gene_id,gene_symbol,utr_sequence,utr_length,gene_id_clean
0,ENST00000000233.5,ENSG00000004059.6,ARF5,CCAGCCAGGGGCAGGCCCCTGATGCCCGGAAGCTCCTGCGTGCATC...,422,ENSG00000004059
1,ENST00000000412.3,ENSG00000003056.3,M6PR,ATTGCACTTTATATGTCCAGCCTCTTCCTCAGTCCCCCAAACCAAA...,1457,ENSG00000003056
2,ENST00000001008.4,ENSG00000004478.5,FKBP4,CCCCTCTCCACCAGCCCTACTCCTGCGGCTGCCTGCCCCCCAGTCT...,2163,ENSG00000004478
3,ENST00000001146.2,ENSG00000003137.4,CYP26B1,CCCAAGACCCACCCGCCTCAGCCCAGCCCAGGCAGCGGGGTGGTGG...,3001,ENSG00000003137
4,ENST00000002125.4,ENSG00000003509.11,NDUFAF7,TATTTCAGCTTGGACATTTTACCCTTCAGTCGGCCCAAGAAATCAA...,810,ENSG00000003509


In [9]:
## Scan every 3'UTR transcript for all siRNA x site-type combinations (counts only)
seed_tx = count_sites(utr_tx, sirna_table)
print(seed_tx.shape)   # expect (28351, 18)
seed_tx.head()

(28351, 18)


,transcript_id,gene_id_clean,siPARP1-A_8mer_count,siPARP1-A_7mer_m8_count,siPARP1-A_7mer_A1_count,siPARP1-A_6mer_count,siPARP1-B_8mer_count,siPARP1-B_7mer_m8_count,siPARP1-B_7mer_A1_count,siPARP1-B_6mer_count,siPARP2-A_8mer_count,siPARP2-A_7mer_m8_count,siPARP2-A_7mer_A1_count,siPARP2-A_6mer_count,siPARP2-B_8mer_count,siPARP2-B_7mer_m8_count,siPARP2-B_7mer_A1_count,siPARP2-B_6mer_count
0,ENST00000000233.5,ENSG00000004059,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,ENST00000000412.3,ENSG00000003056,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0
2,ENST00000001008.4,ENSG00000004478,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0
3,ENST00000001146.2,ENSG00000003137,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,2
4,ENST00000002125.4,ENSG00000003509,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,1


In [10]:
## Aggregate to gene level: count = max over UTR isoforms; present = count > 0; best_site
seed_gene = sites_to_gene(seed_tx, sirna_table)
print(seed_gene.shape)   # expect (19473, 37)

for sirna in sirna_table.index:
    print(sirna, seed_gene[f"{sirna}_best_site"].value_counts().to_dict())

(19473, 37)
siPARP1-A {'none': 12420, '6mer': 2373, '7mer_m8': 1923, '7mer_A1': 1465, '8mer': 1292}
siPARP1-B {'none': 9895, '6mer': 3380, '7mer_A1': 3076, '7mer_m8': 2031, '8mer': 1091}
siPARP2-A {'none': 12500, '6mer': 2619, '7mer_A1': 1751, '7mer_m8': 1515, '8mer': 1088}
siPARP2-B {'none': 10965, '6mer': 4028, '7mer_m8': 2780, '7mer_A1': 1010, '8mer': 690}


In [11]:
## Save gene-level seed annotation and the siRNA table it was built from
out_dir = Path("../../data/datasets/GSE114273/processed")

seed_gene.to_csv(out_dir / "GSE114273_seed_gene.tsv", sep="\t")      # index = gene_id_clean
sirna_table.to_csv(out_dir / "GSE114273_sirna_table.tsv", sep="\t")  # index = sirna

print("Saved:", seed_gene.shape, sirna_table.shape)

Saved: (19473, 37) (4, 8)


#### Summary
- siRNA strands verified against TargetScan ORFs: each sense strand occurs only in its
  intended target (PARP1 or PARP2); no guide strand occurs in any ORF. siPARP1-B corrected
  (deposited sequence had its last two nt swapped).
- All 28,351 human 3'UTR transcripts (TargetScan 7) scanned for 8mer, 7mer-m8, 7mer-A1
  and 6mer sites of each siRNA; aggregated to 19,473 genes ("any isoform": max over UTRs).
- Saved: GSE114273_seed_gene.tsv and GSE114273_sirna_table.tsv.
- Not filtered to expressed genes here; the gene population is set at the DESeq2 stage.